In [1]:
import json
import pickle

def read_json(file_path):    
    with open(file_path) as f:
        json_data = json.load(f)

    return json_data

def read_pickle(file_path):
    with open(file_path, 'rb') as f:
        pickle_data = pickle.load(f)

    return pickle_data

def read_jsonl(filepath):
    data = []
    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data

def save_json(data, filepath, indent=2):
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=indent, ensure_ascii=False)


def save_pickle(data, filepath, protocol=pickle.HIGHEST_PROTOCOL):
    with open(filepath, "wb") as f:
        pickle.dump(data, f, protocol=protocol)

def save_jsonl(data, filepath):
    with open(filepath, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False))
            f.write("\n")

In [2]:
data1 = read_json("../codes/Few_Shot_transformation_and_sampling/fs_tacred_dev_5_way_1_shots_1000episodes_160290.json")
data2 = read_json("../codes/Few_Shot_transformation_and_sampling/fs_tacred_dev_5_way_1_shots_1000episodes_160291.json")
data3 = read_json("../codes/Few_Shot_transformation_and_sampling/fs_tacred_dev_5_way_1_shots_1000episodes_160292.json")

In [3]:
print(len(data1[0]))

1000


In [4]:
core_data = dict()
core_data['shots'] = dict()
core_data['queries'] = dict()
core_data['umbc_shots'] = dict()

def extract_shot(shot):
    new_shot = dict()

    new_shot["id"] = shot["id"]
    new_shot['relation'] = shot['relation']
    new_shot['token'] = shot['token'].copy()
    new_shot['subj_start'] = shot['subj_start']
    new_shot['subj_end'] = shot['subj_end']
    new_shot['obj_start'] = shot['obj_start']
    new_shot['obj_end'] = shot['obj_end']
    new_shot['subj_type'] = shot['subj_type']
    new_shot['obj_type'] = shot['obj_type']
    
    return new_shot

episode_files = []
new_episodes = []

for cur_data in [data1, data2, data3]:
    cur_data = cur_data[0]
    # print(len(cur_data))

    
    for q_ in range(3):
        for episode in cur_data:
            new_episode = dict()

            train_ = episode['meta_train']
            test_ = episode['meta_test'][q_]

            new_ways = []
            for way in train_:
                new_shots = []
                for uidx, shot in enumerate(way):
                    
                    assert shot['token'] == shot['tokens']

                    if uidx > 0:
                        id_ = shot['id']
                        if id_ not in core_data['umbc_shots']:
                            new_shot = extract_shot(shot)
                            core_data['umbc_shots'][id_] = new_shot

                    else:
                        id_ = shot['id']
                        if id_ not in core_data['shots']:
                            new_shot = extract_shot(shot)
                            core_data['shots'][id_] = new_shot
        
                    new_shots.append(id_)
                new_ways.append(new_shots)


            id_ = test_['id']
            if id_ not in core_data['queries']:
                new_shot = extract_shot(test_)
                core_data['queries'][id_] = new_shot

            new_q = [id_]
            new_episode['meta_train'] = new_ways
            new_episode['meta_test'] = new_q

            new_episodes.append(new_episode)

    print(len(new_episodes))

print(len(core_data['shots']))
print(len(core_data['queries']))
print(len(core_data['umbc_shots']))
    

3000
6000
9000
633
7365
0


In [13]:
save_pickle(core_data, "../data/fs_tacred_dev_shots_details.pkl")

In [14]:
save_pickle(new_episodes, "../data/fs_tacred_dev_episodes_1shots.pkl")

In [22]:
test_data = read_pickle("../data/fs_tacred_test_episodes_shots_details.pkl")

In [23]:
test_data_copy = dict()

test_data_copy["shots"] = test_data["shots"]
test_data_copy["queries"] = test_data["queries"]
test_data_copy["umbc_shots"] = dict()

In [24]:
save_pickle(test_data_copy, "../data/fs_tacred_test_episodes_shots_details.pkl")

In [5]:
train_data = read_json("Few_Shot_transformation_and_sampling/data_few_shot/_train_data.json")

In [28]:
test_data = read_json("Few_Shot_transformation_and_sampling/data_few_shot/_test_data.json")

In [6]:
print(type(train_data))

<class 'dict'>


In [27]:
print(train_data.keys())

dict_keys(['no_relation', 'per:employee_of', 'org:alternate_names', 'per:cities_of_residence', 'per:title', 'per:religion', 'org:website', 'per:countries_of_residence', 'org:city_of_headquarters', 'org:members', 'per:spouse', 'org:stateorprovince_of_headquarters', 'org:number_of_employees/members', 'org:subsidiaries', 'org:political/religious_affiliation', 'per:other_family', 'per:stateorprovince_of_birth', 'org:dissolved', 'per:date_of_death', 'org:shareholders', 'per:parents', 'per:cause_of_death', 'per:country_of_birth', 'per:city_of_birth', 'per:charges', 'per:country_of_death'])


In [29]:
print(test_data.keys())

dict_keys(['no_relation', 'org:top_members/employees', 'per:children', 'per:origin', 'org:founded_by', 'per:siblings', 'per:stateorprovinces_of_residence', 'org:member_of', 'per:city_of_death', 'per:date_of_birth', 'per:schools_attended'])


In [7]:
ok = None
for k, v in train_data.items():
    print(k, len(v))
    ok = v[0]

no_relation 59961
per:employee_of 1524
org:alternate_names 808
per:cities_of_residence 374
per:title 2443
per:religion 53
org:website 111
per:countries_of_residence 445
org:city_of_headquarters 382
org:members 170
per:spouse 258
org:stateorprovince_of_headquarters 229
org:number_of_employees/members 75
org:subsidiaries 296
org:political/religious_affiliation 105
per:other_family 179
per:stateorprovince_of_birth 38
org:dissolved 23
per:date_of_death 134
org:shareholders 76
per:parents 152
per:cause_of_death 117
per:country_of_birth 28
per:city_of_birth 65
per:charges 72
per:country_of_death 6


In [8]:
for k,v in ok.items():
    print(k, v)

id 61b3ad44186acb0c25e9
docid APW_ENG_19970216.0744.LDC2007T07
relation per:country_of_death
token ['BEIRUT', ',', 'Lebanon', '(', 'AP', ')', 'Sheik', 'Abbas', 'Musawi', ',', 'Hezbollah', "'s", 'secretary-general', ',', 'his', 'wife', 'and', 'son', 'were', 'killed', 'in', 'February', '1992', 'when', 'Israeli', 'helicopters', 'fired', 'rockets', 'at', 'his', 'car', 'in', 'southern', 'Lebanon', '.']
subj_start 7
subj_end 8
obj_start 32
obj_end 33
subj_type PERSON
obj_type COUNTRY
stanford_pos ['NNP', ',', 'NNP', '-LRB-', 'NNP', '-RRB-', 'NNP', 'NNP', 'NNP', ',', 'NNP', 'POS', 'JJ', ',', 'PRP$', 'NN', 'CC', 'NN', 'VBD', 'VBN', 'IN', 'NNP', 'CD', 'WRB', 'JJ', 'NNS', 'VBD', 'NNS', 'IN', 'PRP$', 'NN', 'IN', 'JJ', 'NNP', '.']
stanford_ner ['LOCATION', 'O', 'LOCATION', 'O', 'ORGANIZATION', 'O', 'PERSON', 'PERSON', 'PERSON', 'O', 'ORGANIZATION', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'DATE', 'DATE', 'O', 'MISC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'LOCATION', 'O']
stanford_head [

In [9]:
from collections import defaultdict
import copy

In [10]:
train_core = defaultdict(list)
train_shots = dict()

for k, v in train_data.items():
    relation = k
    for vv in v:
        vc = extract_shot(vv)
        # vc["relation"] = relation

        train_core[relation].append(vc)
        train_shots[vc["id"]] = vc

print(len(train_shots))
for k, v in train_core.items():
    print(k, len(v))

68124
no_relation 59961
per:employee_of 1524
org:alternate_names 808
per:cities_of_residence 374
per:title 2443
per:religion 53
org:website 111
per:countries_of_residence 445
org:city_of_headquarters 382
org:members 170
per:spouse 258
org:stateorprovince_of_headquarters 229
org:number_of_employees/members 75
org:subsidiaries 296
org:political/religious_affiliation 105
per:other_family 179
per:stateorprovince_of_birth 38
org:dissolved 23
per:date_of_death 134
org:shareholders 76
per:parents 152
per:cause_of_death 117
per:country_of_birth 28
per:city_of_birth 65
per:charges 72
per:country_of_death 6


In [ ]:
train_samples = dict()
train_samples["relation_list"] = list(train_core.keys())
train_samples["in"] =  train_core

In [11]:
save_pickle(train_core, "../data/fs_tacred_train_samples.pkl")

In [43]:
for k in train_samples["relation_list"]:
    print(k)

no_relation
per:employee_of
org:alternate_names
per:cities_of_residence
per:title
per:religion
org:website
per:countries_of_residence
org:city_of_headquarters
org:members
per:spouse
org:stateorprovince_of_headquarters
org:number_of_employees/members
org:subsidiaries
org:political/religious_affiliation
per:other_family
per:stateorprovince_of_birth
org:dissolved
per:date_of_death
org:shareholders
per:parents
per:cause_of_death
per:country_of_birth
per:city_of_birth
per:charges
per:country_of_death
